<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# 🛡️ FinShield Fraud Detection Model Training

This notebook trains a **Random Forest classifier** for invoice fraud detection.

## Model Overview
- **Algorithm**: Random Forest Classifier
- **Input**: 36 numerical features extracted from invoice data
- **Output**: Fraud probability (0.0 - 1.0)
- **Training Data**: JSON dataset with labeled invoices (fraud/legitimate)
- **Invoice Numbers**: Numeric-only (e.g., "12345678"), no alphanumeric formats

## Weights in Fraud Detection Layer
| Component | Weight |
|-----------|--------|
| Duplicate Detection | 40% |
| Vendor Validation | 30% |
| Pattern Analysis | 20% |
| Temporal Checks | 10% |

## Final Score = 60% Rules + 40% ML (this model)

## 1. Import Required Libraries

In [1]:
# Core libraries
import json
import pickle
import joblib
import time
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    classification_report,
    roc_curve, 
    auc,
    roc_auc_score, 
    precision_recall_curve
)
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Load Training Dataset

Load your labeled invoice dataset from JSON file. 

**Expected JSON format:**
```json
[
  {
    "invoiceNumber": "12345678",
    "vendor": "ABC Corp",
    "total": 5000.00,
    "subtotal": 4500.00,
    "tax": 500.00,
    "invoiceDate": "2024-01-15",
    "label": "legitimate"  // or "fraud"
  },
  ...
]
```

**Note:** Invoice numbers should be **numeric-only** (e.g., "12345678"), not alphanumeric (e.g., "INV-001" or "3d762289"). This matches the OCR extraction patterns and fraud detection validation logic.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
# ⚠️ UPDATE THIS PATH to your training dataset
DATASET_PATH = "fraud_training_data.json"

# Load dataset
def load_dataset(path: str) -> pd.DataFrame:
    """Load invoice dataset from JSON file"""
    with open(path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    print(f"✅ Loaded {len(df)} invoices from {path}")
    return df

# Try to load dataset (will fail if file doesn't exist - that's OK for now)
try:
    df = load_dataset(DATASET_PATH)
except FileNotFoundError:
    print(f"⚠️ Dataset not found at: {DATASET_PATH}")
    print("Creating sample synthetic dataset for demonstration...")
    
    # Generate synthetic data for demonstration
    np.random.seed(42)
    n_samples = 2000
    n_fraud = int(n_samples * 0.12)  # 12% fraud rate
    
    # Product descriptions for line items
    products = [
        'Office Supplies', 'Computer Equipment', 'Software License', 
        'Consulting Services', 'Marketing Materials', 'Furniture',
        'Printing Services', 'Maintenance Fee', 'Training Course',
        'Server Hosting', 'Cloud Storage', 'IT Support'
    ]
    
    # Generate legitimate invoices
    legitimate = []
    for i in range(n_samples - n_fraud):
        num_items = np.random.randint(2, 7)
        line_items = []
        subtotal = 0
        
        for j in range(num_items):
            qty = np.random.randint(1, 20)
            unit_price = round(np.random.uniform(10, 500), 2)
            amount = round(qty * unit_price, 2)
            subtotal += amount
            
            line_items.append({
                'description': np.random.choice(products),
                'quantity': qty,
                'unit_price': unit_price,
                'amount': amount
            })
        
        tax = round(subtotal * 0.12, 2)
        total = round(subtotal + tax, 2)
        
        # ✅ UPDATED: Use numeric-only invoice numbers (8 digits)
        legitimate.append({
            'invoiceNumber': str(np.random.randint(10000000, 99999999)),
            'issuedTo': np.random.choice(['ABC Corp', 'XYZ Ltd', 'Acme Inc', 'Global Services', 'Tech Solutions']),
            'totalAmount': total,
            'subtotalAmount': subtotal,
            'taxAmount': tax,
            'invoiceDate': f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}',
            'lineItems': line_items,
            'reviewDecision': 'approved'
        })
    
    # Generate fraudulent invoices
    fraud = []
    for i in range(n_fraud):
        # Fraudulent invoices often have fewer items, calculation errors, or no items
        num_items = np.random.randint(0, 3)
        line_items = []
        subtotal = 0
        
        for j in range(num_items):
            qty = np.random.randint(1, 200)  # Sometimes suspiciously high
            unit_price = round(np.random.uniform(50, 2000), 2)  # Often round numbers
            # Intentional calculation errors for fraud
            if np.random.random() < 0.4:
                amount = round(qty * unit_price * np.random.uniform(0.8, 1.3), 2)  # Wrong calculation
            else:
                amount = round(qty * unit_price, 2)
            subtotal += amount
            
            line_items.append({
                'description': np.random.choice(products + ['Misc', 'Various Items', '']),
                'quantity': qty,
                'unit_price': unit_price,
                'amount': amount
            })
        
        # Fraud often has round total amounts
        total = np.random.choice([1000, 5000, 10000, 50000]) + (np.random.uniform(0, 100) if np.random.random() < 0.3 else 0)
        tax = round(total * 0.12, 2)
        subtotal = round(total - tax, 2)
        
        # ✅ UPDATED: Use numeric-only invoice numbers or missing (no alphanumeric like 'test' or 'INV-xxx')
        # Fraudulent patterns: very short numbers, missing, or empty
        inv_choice = np.random.choice(['short', 'missing', 'normal'])
        if inv_choice == 'short':
            invoice_num = str(np.random.randint(100, 999))  # Suspiciously short (3 digits)
        elif inv_choice == 'missing':
            invoice_num = ''  # Missing invoice number
        else:
            invoice_num = str(np.random.randint(10000000, 99999999))  # Normal 8-digit
        
        fraud.append({
            'invoiceNumber': invoice_num,
            'issuedTo': np.random.choice(['Unknown Vendor', 'New Supplier', '', 'CASH', 'misc']),
            'totalAmount': total,
            'subtotalAmount': subtotal,
            'taxAmount': tax,
            'invoiceDate': f'2025-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}',  # Future dates
            'lineItems': line_items,
            'reviewDecision': 'rejected'
        })
    
    # Combine
    all_data = legitimate + fraud
    df = pd.DataFrame(all_data)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
    
    # Save synthetic dataset
    with open(DATASET_PATH, 'w') as f:
        json.dump(all_data, f, indent=2)
    print(f"✅ Created synthetic dataset with {len(df)} invoices ({n_fraud} fraud, {n_samples-n_fraud} legitimate)")
    print(f"   Note: Invoice numbers are now NUMERIC-ONLY (8 digits or missing for fraud)")

df.head(10)

JSONDecodeError: Expecting value: line 61047 column 20 (char 1399922)

## 3. Exploratory Data Analysis

In [ ]:
# Dataset statistics
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"\nTotal invoices: {len(df)}")
print(f"\nClass distribution:")
print(df['reviewDecision'].value_counts())
print(f"\nClass percentages:")
print(df['reviewDecision'].value_counts(normalize=True) * 100)

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution pie chart
df['reviewDecision'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], colors=['#4CAF50', '#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

# Amount distribution by class
for label in df['reviewDecision'].unique():
    subset = df[df['reviewDecision'] == label]['totalAmount']
    axes[1].hist(subset, bins=30, alpha=0.6, label=label)
axes[1].set_xlabel('Invoice Amount')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Amount Distribution by Class')
axes[1].legend()

plt.tight_layout()
plt.show()

# Basic statistics
print("\n" + "=" * 50)
print("NUMERICAL STATISTICS")
print("=" * 50)
df.describe()

## 4. Feature Engineering

Extract numerical features from invoice data for the Random Forest model.

**Updated to include Quantity & Unit Price features (36 total features)**

**Important:** 
- Invoice numbers are expected to be **numeric-only** (e.g., "12345678")
- The `inv_num_is_numeric` feature gives higher scores to numeric-only invoice numbers
- This aligns with the OCR extraction patterns that only capture numeric invoice numbers

In [ ]:
def extract_features(invoice: dict) -> dict:
    """
    Extract numerical features from an invoice for fraud detection.
    
    Features extracted (36 total):
    - Amount features (total, subtotal, tax, ratios)
    - Pattern features (round numbers, invoice number format)
    - Temporal features (date parsing, age)
    - Completeness features (missing fields)
    - Line item features (count, quantities, unit prices, calculation checks)
    
    Note: Invoice numbers are expected to be NUMERIC-ONLY (e.g., "12345678").
          The 'inv_num_is_numeric' feature detects alphanumeric formats as suspicious.
    """
    features = {}
    
    # ===== AMOUNT FEATURES =====
    total = float(invoice.get('totalAmount', 0) or 0)
    subtotal = float(invoice.get('subtotalAmount', 0) or 0)
    tax = float(invoice.get('taxAmount', 0) or 0)
    
    features['total'] = total
    features['subtotal'] = subtotal
    features['tax'] = tax
    features['tax_ratio'] = tax / total if total > 0 else 0
    features['amount_log'] = np.log1p(total)  # Log transform for skewed amounts
    
    # Round number detection (suspicious pattern)
    features['is_round_100'] = 1 if total % 100 == 0 else 0
    features['is_round_1000'] = 1 if total % 1000 == 0 else 0
    features['is_round_500'] = 1 if total % 500 == 0 else 0
    
    # Decimal analysis
    decimal_part = total - int(total)
    features['has_cents'] = 1 if decimal_part > 0 else 0
    features['cents_value'] = decimal_part
    
    # ===== INVOICE NUMBER FEATURES =====
    # NOTE: Invoice numbers should be numeric-only (e.g., "12345678")
    # Alphanumeric formats like "INV-001" will have inv_num_is_numeric = 0
    inv_num = str(invoice.get('invoiceNumber', '') or '')
    features['inv_num_length'] = len(inv_num)
    features['inv_num_has_prefix'] = 1 if any(inv_num.upper().startswith(p) for p in ['INV', 'INVOICE', 'PO', 'REF']) else 0
    features['inv_num_is_numeric'] = 1 if inv_num.isdigit() else 0
    features['inv_num_missing'] = 1 if not inv_num.strip() else 0
    
    # ===== ISSUED TO FEATURES =====
    issued_to = str(invoice.get('issuedTo', '') or '')
    features['issued_to_length'] = len(issued_to)
    features['issued_to_missing'] = 1 if not issued_to.strip() else 0
    features['issued_to_is_generic'] = 1 if issued_to.lower() in ['cash', 'unknown', 'misc', 'various', ''] else 0
    
    # ===== DATE FEATURES =====
    date_str = str(invoice.get('invoiceDate', '') or '')
    features['date_missing'] = 1 if not date_str.strip() else 0
    
    try:
        invoice_date = datetime.strptime(date_str, '%Y-%m-%d')
        today = datetime.now()
        
        features['days_old'] = (today - invoice_date).days
        features['is_future'] = 1 if invoice_date > today else 0
        features['is_weekend'] = 1 if invoice_date.weekday() >= 5 else 0
        features['month'] = invoice_date.month
        features['day_of_week'] = invoice_date.weekday()
        features['is_month_end'] = 1 if invoice_date.day >= 28 else 0
    except:
        features['days_old'] = -1  # Invalid date marker
        features['is_future'] = 0
        features['is_weekend'] = 0
        features['month'] = 0
        features['day_of_week'] = -1
        features['is_month_end'] = 0
    
    # ===== LINE ITEMS FEATURES =====
    line_items = invoice.get('lineItems', [])
    line_count = len(line_items) if line_items else 0
    features['line_item_count'] = line_count
    features['has_line_items'] = 1 if line_count > 0 else 0
    features['avg_line_item_value'] = total / line_count if line_count > 0 else total
    
    # ===== QUANTITY & UNIT PRICE FEATURES =====
    quantities = []
    unit_prices = []
    calculation_errors = 0
    
    for item in line_items:
        qty = item.get('quantity', 1)
        unit_price = item.get('unit_price', item.get('amount', 0))
        item_amount = item.get('amount', 0)
        
        quantities.append(qty)
        unit_prices.append(unit_price)
        
        # Check calculation: qty × unit_price ≈ amount
        expected = qty * unit_price
        if abs(expected - item_amount) > 0.02:
            calculation_errors += 1
    
    features['max_quantity'] = max(quantities) if quantities else 1
    features['avg_quantity'] = np.mean(quantities) if quantities else 1
    features['max_unit_price'] = max(unit_prices) if unit_prices else total
    features['avg_unit_price'] = np.mean(unit_prices) if unit_prices else total
    features['calculation_mismatches'] = calculation_errors
    
    # Single item dominance
    if line_items and total > 0:
        max_item_amount = max(item.get('amount', 0) for item in line_items)
        features['single_item_dominance'] = max_item_amount / total
    else:
        features['single_item_dominance'] = 1.0
    
    features['has_high_quantity'] = 1 if features['max_quantity'] > 100 else 0
    
    # ===== COMPLETENESS SCORE =====
    required_fields = ['invoiceNumber', 'issuedTo', 'totalAmount', 'invoiceDate']
    missing_count = sum(1 for f in required_fields if not invoice.get(f))
    features['missing_fields_count'] = missing_count
    features['completeness_score'] = 1 - (missing_count / len(required_fields))
    
    return features


# Test feature extraction on one row
sample_invoice = df.iloc[0].to_dict()
print("Sample invoice:")
print(json.dumps(sample_invoice, indent=2, default=str))
print("\nExtracted features:")
sample_features = extract_features(sample_invoice)
for k, v in sample_features.items():
    print(f"  {k}: {v}")

In [ ]:
# Extract features for all invoices
print("Extracting features for all invoices...")

feature_list = []
for idx, row in df.iterrows():
    features = extract_features(row.to_dict())
    features['reviewDecision'] = row['reviewDecision']  # Keep label for training
    feature_list.append(features)

# Create features DataFrame
df_features = pd.DataFrame(feature_list)

print(f"✅ Extracted {len(df_features.columns) - 1} features from {len(df_features)} invoices")
print(f"\nFeature columns (36 features):")
print([c for c in df_features.columns if c != 'reviewDecision'])

df_features.head()

## 5. Prepare Data for Training

In [ ]:
# Prepare X (features) and y (labels)
feature_columns = [c for c in df_features.columns if c != 'reviewDecision']
X = df_features[feature_columns]
y = (df_features['reviewDecision'] == 'rejected').astype(int)  # 1 = fraud (rejected), 0 = legitimate (approved)

print(f"Feature matrix shape: {X.shape}")
print(f"Label vector shape: {y.shape}")
print(f"Total features: {len(feature_columns)}")
print(f"\nLabel distribution:")
print(f"  Legitimate (0): {(y == 0).sum()}")
print(f"  Fraud (1): {(y == 1).sum()}")
print(f"  Fraud rate: {y.mean() * 100:.2f}%")

# Train/Test split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Maintain class proportions
)

print(f"\n✅ Train/Test Split Complete:")
print(f"  Training set: {len(X_train)} samples ({(y_train == 1).sum()} fraud)")
print(f"  Test set: {len(X_test)} samples ({(y_test == 1).sum()} fraud)")

In [ ]:
# Helper for production-style vector extraction (used by simulation below)
def extract_feature_vector(invoice: dict) -> list:
    if 'feature_columns' not in globals():
        raise NameError("feature_columns not defined. Run the feature extraction + training prep cells first.")
    features = extract_features(invoice)
    return [features.get(name, 0) for name in feature_columns]


## 6. Train Random Forest Model

Using the agreed configuration:
- **n_estimators**: 100 (number of trees)
- **max_depth**: 10 (prevent overfitting)
- **class_weight**: 'balanced' (handle imbalanced fraud data)
- **random_state**: 42 (reproducibility)

In [ ]:
# Initialize Random Forest with optimal parameters
model = RandomForestClassifier(
    n_estimators=100,        # 100 trees for good accuracy
    max_depth=10,            # Limit depth to prevent overfitting
    class_weight='balanced', # Handle imbalanced data (fraud is rare)
    random_state=42,         # Reproducibility
    n_jobs=-1,               # Use all CPU cores
    min_samples_split=5,     # Minimum samples to split a node
    min_samples_leaf=2       # Minimum samples per leaf
)

# Train the model
print("🔄 Training Random Forest model...")
start_time = time.time()

model.fit(X_train, y_train)

training_time = time.time() - start_time
print(f"✅ Model trained in {training_time:.2f} seconds")
print(f"\nModel parameters:")
print(model.get_params())

## 7. Model Evaluation

In [ ]:
# Predictions on test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of fraud class

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"\n📊 Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"🎯 Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"🔍 Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"⚖️  F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"📈 ROC-AUC:   {roc_auc:.4f} ({roc_auc*100:.2f}%)")

# Interpretation
print("\n" + "=" * 50)
print("INTERPRETATION")
print("=" * 50)
print(f"• Precision: Of all invoices flagged as fraud, {precision*100:.1f}% are actually fraudulent")
print(f"• Recall: We catch {recall*100:.1f}% of all actual fraudulent invoices")
print(f"• F1-Score: Balanced measure of precision and recall")
print(f"• ROC-AUC: Model's ability to distinguish between classes")

In [ ]:
# Confusion Matrix
print("\nCONFUSION MATRIX")
print("-" * 30)
cm = confusion_matrix(y_test, y_pred)
print(f"\n                 Predicted")
print(f"              Legit  Fraud")
print(f"Actual Legit   {cm[0][0]:4d}   {cm[0][1]:4d}")
print(f"Actual Fraud   {cm[1][0]:4d}   {cm[1][1]:4d}")

# Visualize confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "=" * 50)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

## 8. Feature Importance Analysis

Understanding which features contribute most to fraud detection.

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=" * 50)
print("TOP 15 MOST IMPORTANT FEATURES")
print("=" * 50)
for i, row in feature_importance.head(15).iterrows():
    bar = "█" * int(row['importance'] * 100)
    print(f"{row['feature']:25s} {row['importance']:.4f} {bar}")

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(20)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_features)))[::-1]

bars = ax.barh(range(len(top_features)), top_features['importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances for Fraud Detection')
ax.grid(True, axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, top_features['importance'])):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Feature importance summary
print("\n💡 KEY INSIGHTS:")
top_3 = feature_importance.head(3)['feature'].tolist()
print(f"   Top predictors: {', '.join(top_3)}")
print(f"   These features contribute {feature_importance.head(3)['importance'].sum()*100:.1f}% to predictions")

## 9. Save Model

Save the trained model for use in the fraud detection layer.

In [ ]:
# Model metadata
model_metadata = {
    "model_name": "fraud_detection_rf",
    "version": "1.0.0",
    "algorithm": "RandomForestClassifier",
    "trained_at": datetime.now().isoformat(),
    "training_samples": len(X_train),
    "test_samples": len(X_test),
    "n_features": len(feature_columns),
    "feature_names": feature_columns,
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc)
    },
    "hyperparameters": {
        "n_estimators": 100,
        "max_depth": 10,
        "class_weight": "balanced",
        "min_samples_split": 5,
        "min_samples_leaf": 2
    },
    "class_distribution": {
        "training_fraud": int((y_train == 1).sum()),
        "training_legitimate": int((y_train == 0).sum()),
        "test_fraud": int((y_test == 1).sum()),
        "test_legitimate": int((y_test == 0).sum())
    }
}

# Save model to pickle
MODEL_PATH = "fraud_model_rf.pkl"
METADATA_PATH = "fraud_model_metadata.json"

# Save model
joblib.dump(model, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

# Save metadata
with open(METADATA_PATH, 'w') as f:
    json.dump(model_metadata, f, indent=2)
print(f"✅ Metadata saved to: {METADATA_PATH}")

# Check file size
import os
model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"\n📦 Model file size: {model_size_mb:.2f} MB")

if model_size_mb > 50:
    print("⚠️ Warning: Model size is large. Consider reducing n_estimators or max_depth.")
elif model_size_mb < 20:
    print("✅ Model size is within expected range (~15-20MB)")

## 10. Test Model Loading

Verify the saved model can be loaded and used for predictions.

In [ ]:
# Load and test model
print("Testing model loading...")

# Load model
loaded_model = joblib.load(MODEL_PATH)

# Load metadata
with open(METADATA_PATH, 'r') as f:
    loaded_metadata = json.load(f)

print(f"✅ Model loaded: {loaded_metadata['model_name']} v{loaded_metadata['version']}")
print(f"   Trained at: {loaded_metadata['trained_at']}")
print(f"   Features: {loaded_metadata['n_features']}")

# Test prediction on sample
sample_features = X_test.iloc[:5]
predictions = loaded_model.predict(sample_features)
probabilities = loaded_model.predict_proba(sample_features)[:, 1]

print("\n📊 Sample predictions:")
print("-" * 50)
for i, (pred, prob, actual) in enumerate(zip(predictions, probabilities, y_test.iloc[:5])):
    status = "✓" if pred == actual else "✗"
    label = "FRAUD" if pred == 1 else "LEGIT"
    actual_label = "FRAUD" if actual == 1 else "LEGIT"
    print(f"  Invoice {i+1}: {label} (confidence: {prob*100:.1f}%) | Actual: {actual_label} {status}")

## 11. Export Feature Extractor

Export the feature extraction function for use in the fraud detection layer.

**Updated to include quantity and unit price features (36 total)**

**Important:** Invoice numbers must be **numeric-only** (e.g., "12345678"). The `inv_num_is_numeric` feature will flag alphanumeric formats as suspicious.

In [ ]:
# Export feature extractor as a Python module
FEATURE_EXTRACTOR_CODE = '''"""
Feature Extractor for Fraud Detection Model
Auto-generated from training notebook - 36 features including qty/unit_price

IMPORTANT: Invoice numbers must be NUMERIC-ONLY (e.g., "12345678")
           Alphanumeric formats like "INV-001" or "3d762289" are NOT supported
           and will be flagged by the inv_num_is_numeric feature
"""

import numpy as np
from datetime import datetime
from typing import Dict, Any, List

# Feature names expected by the model (must match training order)
FEATURE_NAMES = {feature_names}

def extract_features(invoice: Dict[str, Any]) -> Dict[str, float]:
    """
    Extract 36 numerical features from an invoice for fraud detection.
    
    Note: Invoice numbers should be numeric-only (e.g., "12345678").
          The inv_num_is_numeric feature detects non-numeric formats as suspicious.
    """
    features = {{}}
    
    # ===== AMOUNT FEATURES =====
    total = float(invoice.get('totalAmount', 0) or invoice.get('total', 0) or 0)
    subtotal = float(invoice.get('subtotalAmount', 0) or invoice.get('subtotal', 0) or 0)
    tax = float(invoice.get('taxAmount', 0) or invoice.get('tax', 0) or 0)
    
    features['total'] = total
    features['subtotal'] = subtotal
    features['tax'] = tax
    features['tax_ratio'] = tax / total if total > 0 else 0
    features['amount_log'] = np.log1p(total)
    
    features['is_round_100'] = 1 if total % 100 == 0 else 0
    features['is_round_1000'] = 1 if total % 1000 == 0 else 0
    features['is_round_500'] = 1 if total % 500 == 0 else 0
    
    decimal_part = total - int(total)
    features['has_cents'] = 1 if decimal_part > 0 else 0
    features['cents_value'] = decimal_part
    
    # ===== INVOICE NUMBER FEATURES =====
    # NOTE: Expected to be numeric-only (e.g., "12345678")
    inv_num = str(invoice.get('invoiceNumber', '') or '')
    features['inv_num_length'] = len(inv_num)
    features['inv_num_has_prefix'] = 1 if any(inv_num.upper().startswith(p) for p in ['INV', 'INVOICE', 'PO', 'REF']) else 0
    features['inv_num_is_numeric'] = 1 if inv_num.isdigit() else 0
    features['inv_num_missing'] = 1 if not inv_num.strip() else 0
    
    # ===== ISSUED TO FEATURES =====
    issued_to = str(invoice.get('issuedTo', '') or '')
    features['issued_to_length'] = len(issued_to)
    features['issued_to_missing'] = 1 if not issued_to.strip() else 0
    features['issued_to_is_generic'] = 1 if issued_to.lower() in ['cash', 'unknown', 'misc', 'various', ''] else 0
    
    # ===== DATE FEATURES =====
    date_str = str(invoice.get('invoiceDate', '') or '')
    features['date_missing'] = 1 if not date_str.strip() else 0
    
    try:
        invoice_date = datetime.strptime(date_str, '%Y-%m-%d')
        today = datetime.now()
        
        features['days_old'] = (today - invoice_date).days
        features['is_future'] = 1 if invoice_date > today else 0
        features['is_weekend'] = 1 if invoice_date.weekday() >= 5 else 0
        features['month'] = invoice_date.month
        features['day_of_week'] = invoice_date.weekday()
        features['is_month_end'] = 1 if invoice_date.day >= 28 else 0
    except:
        features['days_old'] = -1
        features['is_future'] = 0
        features['is_weekend'] = 0
        features['month'] = 0
        features['day_of_week'] = -1
        features['is_month_end'] = 0
    
    # ===== LINE ITEMS FEATURES =====
    line_items = invoice.get('lineItems', [])
    line_count = len(line_items) if line_items else 0
    features['line_item_count'] = line_count
    features['has_line_items'] = 1 if line_count > 0 else 0
    features['avg_line_item_value'] = total / line_count if line_count > 0 else total
    
    # ===== QUANTITY & UNIT PRICE FEATURES =====
    quantities = []
    unit_prices = []
    calculation_errors = 0
    
    for item in line_items:
        qty = item.get('quantity', 1)
        unit_price = item.get('unit_price', item.get('amount', 0))
        item_amount = item.get('amount', 0)
        
        quantities.append(qty)
        unit_prices.append(unit_price)
        
        expected = qty * unit_price
        if abs(expected - item_amount) > 0.02:
            calculation_errors += 1
    
    features['max_quantity'] = max(quantities) if quantities else 1
    features['avg_quantity'] = np.mean(quantities) if quantities else 1
    features['max_unit_price'] = max(unit_prices) if unit_prices else total
    features['avg_unit_price'] = np.mean(unit_prices) if unit_prices else total
    features['calculation_mismatches'] = calculation_errors
    
    if line_items and total > 0:
        max_item_amount = max(item.get('amount', 0) for item in line_items)
        features['single_item_dominance'] = max_item_amount / total
    else:
        features['single_item_dominance'] = 1.0
    
    features['has_high_quantity'] = 1 if features['max_quantity'] > 100 else 0
    
    # ===== COMPLETENESS SCORE =====
    required_fields = ['invoiceNumber', 'issuedTo', 'totalAmount', 'invoiceDate']
    missing_count = sum(1 for f in required_fields if not invoice.get(f))
    features['missing_fields_count'] = missing_count
    features['completeness_score'] = 1 - (missing_count / len(required_fields))
    
    return features


def extract_feature_vector(invoice: Dict[str, Any]) -> List[float]:
    """Extract features as an ordered list matching model training order."""
    features = extract_features(invoice)
    return [features.get(name, 0) for name in FEATURE_NAMES]
'''.format(feature_names=feature_columns)

# Save feature extractor
EXTRACTOR_PATH = "feature_extractor.py"
with open(EXTRACTOR_PATH, 'w') as f:
    f.write(FEATURE_EXTRACTOR_CODE)

print(f"✅ Feature extractor saved to: {EXTRACTOR_PATH}")
print(f"   Total features: 36 (includes qty/unit_price)")
print(f"   Invoice numbers: NUMERIC-ONLY (e.g., '12345678')")
print(f"\nUsage in fraud detection layer:")
print("  from feature_extractor import extract_feature_vector")
print("  features = extract_feature_vector(invoice_data)")
print("  prediction = model.predict([features])")

## 12. Upload Model to S3

Uploads the trained fraud model and its metadata JSON to AWS S3 using the same
`save_model_to_s3()` utility used by the production scheduler. This also invalidates
the Redis and local model caches across all workers so the new model is picked up
on the next `/ocr` request.

**S3 Key**: `models/fraud_model_rf.pkl`  
**Metadata Key**: `models/fraud_model_rf_metadata.json`

In [ ]:
# ── Upload fraud model to S3 ─────────────────────────────────────────────
import sys, os

# Ensure the AI_SERVICE root is on the path so app.* imports work
AI_SERVICE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AI_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, AI_SERVICE_ROOT)

from app.core.constants import FRAUD_S3_MODEL_KEY, FRAUD_S3_METADATA_KEY, FRAUD_FEATURE_NAMES
from app.engines.fraud.model_loader import save_fraud_model_to_s3_explicit, clear_fraud_model_cache
from datetime import datetime

# Safely read metric variables (default to 0 if evaluation cell wasn't run) and cast to float/int
def _safe_float(name, default=0.0):
    val = globals().get(name, default)
    try: return float(val)
    except: return default

def _safe_int(name, default=0):
    val = globals().get(name, default)
    try: return int(val)
    except: return default

# We also cast hyperparameters to native int/float just in case numpy types crept in
s3_metadata = {
    'model_name': 'fraud_detection_rf',
    'version': datetime.now().strftime('%Y.%m.%d'),
    'algorithm': 'RandomForestClassifier',
    'trained_at': datetime.now().isoformat(),
    'feature_names': FRAUD_FEATURE_NAMES,
    'n_features': len(FRAUD_FEATURE_NAMES),
    'hyperparameters': {
        'n_estimators': _safe_int('model.n_estimators', getattr(model, 'n_estimators', 100)),
        'max_depth': _safe_int('model.max_depth', getattr(model, 'max_depth', 10)),
        'class_weight': 'balanced',
    },
    'metrics': {
        'accuracy':  round(_safe_float('accuracy'),  4),
        'precision': round(_safe_float('precision'), 4),
        'recall':    round(_safe_float('recall'),    4),
        'f1':        round(_safe_float('f1'),        4),
        'roc_auc':   round(_safe_float('roc_auc'),   4),
    },
}

print(f'Uploading model to S3 key: {FRAUD_S3_MODEL_KEY} ...')
try:
    bucket = os.environ.get('MODEL_BUCKET_NAME', '<MODEL_BUCKET_NAME>')
    save_fraud_model_to_s3_explicit(model, metadata=s3_metadata)
    clear_fraud_model_cache()
    print(f'✅  Model uploaded  → s3://{bucket}/{FRAUD_S3_MODEL_KEY}')
    print(f'    Metadata        → s3://{bucket}/{FRAUD_S3_METADATA_KEY}')
    print('    Redis + local model caches invalidated across all workers.')
except Exception as e:
    print(f'❌  S3 upload failed: {e}')
    print('    Check that AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION, MODEL_BUCKET_NAME are set in .env')


## 13. Summary

### Training Complete! 🎉

**Model Files Generated:**
- `fraud_model_rf.pkl` - Trained Random Forest model
- `fraud_model_metadata.json` - Model metadata and metrics
- `feature_extractor.py` - Feature extraction module

### Important Notes:
- **Invoice numbers must be numeric-only** (e.g., "12345678")
- The system does NOT accept alphanumeric invoice numbers (e.g., "INV-001", "3d762289")
- This aligns with OCR extraction patterns: `\d{3,}` (3+ digits only)
- Training data generated by `generate_data.py` uses 8-digit numeric invoice numbers

### Next Steps:
1. Copy model files to `app/engines/fraud/` or upload to S3
2. Create a model loader in the fraud detection layer
3. Use the feature extractor to process invoices before prediction

### Retraining Schedule:
- Retrain monthly after collecting 500+ new auditor-reviewed invoices
- Export training data from MongoDB where `reviewDecision` is set
- Maintain at least 25+ fraud examples in training data
- Ensure invoice numbers in training data are numeric-only

In [ ]:
# Final summary output
print("=" * 60)
print("           FRAUD DETECTION MODEL - TRAINING COMPLETE")
print("=" * 60)
print(f"""
📁 FILES GENERATED:
   • fraud_model_rf.pkl ({model_size_mb:.2f} MB)
   • fraud_model_metadata.json
   • feature_extractor.py
   
📊 MODEL PERFORMANCE:
   • Accuracy:  {accuracy*100:.2f}%
   • Precision: {precision*100:.2f}%
   • Recall:    {recall*100:.2f}%
   • F1-Score:  {f1*100:.2f}%
   • ROC-AUC:   {roc_auc*100:.2f}%
   
🔢 TRAINING DATA:
   • Total samples: {len(df)}
   • Fraud samples: {(y == 1).sum()}
   • Features: {len(feature_columns)}

✅ Ready for integration with fraud detection layer!
""")
print("=" * 60)

## 8. Fraud Layer Simulation & Verification

This section simulates the **actual fraud detection layer** used in the production application.
It selects raw invoices (before any processing), extracts features using the exact same logic as production, and runs the model.

### Goals:
1. Verify `extract_feature_vector()` works correctly on raw JSON data
2. Confirm model accuracy on end-to-end flow
3. Show probability scores for specific examples

In [ ]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning

def simulate_fraud_layer(model, df_raw, n_samples=5, show_details=False):
    """
    Simulates the production fraud detection process.
    
    Args:
        model: Trained Random Forest model
        df_raw: Original DataFrame containing raw invoice data (before feature extraction)
        n_samples: Number of random samples to test
        show_details: If True, shows line items and key feature values
    """
    print("=" * 80)
    print(f"FRAUD LAYER SIMULATION (Testing {n_samples} random invoices)")
    print("=" * 80)
    
    # Select random samples
    samples = df_raw.sample(n=n_samples, random_state=42)
    
    correct_predictions = 0
    
    print(f"{'-'*80}")
    print(f"{'INVOICE':<12} | {'VENDOR':<20} | {'TOTAL':>10} | {'ACTUAL':<10} | {'PREDICTED':<10} | {'PROB':>6}")
    print(f"{'-'*80}")
    
    for idx, row in samples.iterrows():
        # 1. Convert row to dict (Simulate receiving JSON payload)
        invoice_json = row.to_dict()
        
        # 2. Extract Feature Vector (The crucial production step)
        features = extract_feature_vector(invoice_json)
        
        # 3. Predict (suppress sklearn warnings about feature names)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
            features_reshaped = [features]
            prob = model.predict_proba(features_reshaped)[0][1]
            prediction = 1 if prob >= 0.5 else 0
        
        # 4. Compare
        actual_label = 1 if row['reviewDecision'] == 'rejected' else 0
        status_icon = "✅" if prediction == actual_label else "❌"
        
        if prediction == actual_label:
            correct_predictions += 1
            
        # Formatting output
        inv_num = str(row.get('invoiceNumber', ''))[:10]
        vendor = str(row.get('issuedTo', ''))[:18]
        total = f"${row.get('totalAmount', 0):,.2f}"
        actual_str = "FRAUD" if actual_label else "LEGIT"
        pred_str = "FRAUD" if prediction else "LEGIT"
        
        print(f"{inv_num:<12} | {vendor:<20} | {total:>10} | {actual_str:<10} | {pred_str:<10} | {prob:.1%}")
        
        # Show detailed breakdown if requested
        if show_details:
            print(f"\n  📋 DETAILED BREAKDOWN:")
            
            # Line items
            line_items = row.get('lineItems', [])
            if line_items:
                print(f"  Line Items ({len(line_items)} items):")
                for i, item in enumerate(line_items[:3], 1):  # Show first 3 items
                    qty = item.get('quantity', 1)
                    unit_price = item.get('unit_price', item.get('amount', 0))
                    amount = item.get('amount', 0)
                    desc = str(item.get('description', 'N/A'))[:30]
                    calc_check = "✓" if abs(qty * unit_price - amount) <= 0.02 else "✗ MISMATCH"
                    print(f"    {i}. {desc:<30} | Qty: {qty:>4} × ${unit_price:>8.2f} = ${amount:>8.2f} {calc_check}")
                if len(line_items) > 3:
                    print(f"    ... and {len(line_items) - 3} more items")
            else:
                print(f"  Line Items: None (⚠️ SUSPICIOUS)")
            
            # Key features extracted
            feature_dict = extract_features(invoice_json)
            print(f"\n  🔍 KEY FEATURES:")
            print(f"    • Invoice Date: {row.get('invoiceDate', 'N/A')} (Days old: {feature_dict.get('days_old', 'N/A')})")
            print(f"    • Tax Ratio: {feature_dict.get('tax_ratio', 0):.2%} (Expected ~12%)")
            print(f"    • Round Amount: ${row.get('totalAmount', 0):,.2f} " + 
                  f"({'✗ Round $1000' if feature_dict.get('is_round_1000') else '✓ Has cents' if feature_dict.get('has_cents') else '✗ Round $100'})")
            print(f"    • Invoice Number: '{row.get('invoiceNumber', '')}' " + 
                  f"({'✓ Has prefix' if feature_dict.get('inv_num_has_prefix') else '✗ No prefix'})")
            print(f"    • Vendor: '{row.get('issuedTo', '')}' " + 
                  f"({'✗ Generic name' if feature_dict.get('issued_to_is_generic') else '✓ Specific'})")
            print(f"    • Completeness: {feature_dict.get('completeness_score', 0):.0%}")
            print(f"    • Calculation Mismatches: {feature_dict.get('calculation_mismatches', 0)}")
            
            print(f"\n  {status_icon} RESULT: {pred_str} (confidence: {prob:.1%})\n")
            print(f"{'-'*80}")
        
    print(f"{'-'*80}")
    accuracy = (correct_predictions / n_samples) * 100
    print(f"\n🎯 Simulation Accuracy: {accuracy:.1f}%")
    
    if accuracy == 100:
        print("✅ The feature extraction and model prediction logic is CONSISTENT with training!")
    else:
        print("⚠️ Discrepancy detected. Check feature extraction logic.")

# Run the simulation - first without details, then with details for 3 examples
print("QUICK OVERVIEW (10 invoices):\n")
try:
    simulate_fraud_layer(model, df, n_samples=10, show_details=False)
    
    print("\n\n" + "="*80)
    print("DETAILED BREAKDOWN (3 invoices):")
    print("="*80 + "\n")
    simulate_fraud_layer(model, df, n_samples=3, show_details=True)
    
except NameError as e:
    print(f"Error: {e}. Make sure you have run the cells defining 'model', 'df', and 'extract_feature_vector'!")
